[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/MFM/blob/main/Quantlets/Ch_15/MFM_ch15_colour_words/MFM_ch15_colour_words.ipynb)

# MFM_ch15_colour_words

Second case study of Chapter 15: Garcia, Hu and Rohrer (2023), *The colour of finance words*, Journal of Financial Economics 147(3), 525-549 ([doi:10.1016/j.jfineco.2022.11.006](https://doi.org/10.1016/j.jfineco.2022.11.006)), replicated on the FNSPID headlines of the chapter. Positive and negative word lists are learned from price reactions with the robust multinomial inverse regression (MNIR) of the paper: 500 subsamples of 5,000 stock-days, a Poisson inverse regression of every word count on the standardised 4-day excess return (close of day d-1 to close of day d+2, minus SPY), and the net share D+ of subsamples in which the word loads positively; words with D+ >= 80% (<= -80%) form the positive (negative) list.

The lists are learned on the stock-days up to December 2019 and evaluated out of sample on 2020-2023, against the Loughran-McDonald (LM) lists and the saved FinBERT and Qwen2.5-7B scores of the chapter (no language model is run). Differences from the paper: headlines instead of earnings calls; the sign of each loading is kept when the score statistic passes the AIC rule (z^2 > 2) instead of a gamma-lasso path; no control variables or fixed effects; the names of the 16 firms are removed instead of a part-of-speech filter for proper nouns; word shares are not winsorized (they lie in [0, 1] and are 0 on most stock-days). Standard errors are clustered by day; R2 differences use a day-cluster bootstrap (999 draws).

*Modelling Financial Markets (MFM), Chapter 15: LLMs and Sentiment Analysis. Author: Daniel Traian Pele.*

> **Working in Google Colab? Save your own copy first.**
> The Colab link opens a temporary copy: when you close the tab or the session times out, your changes are lost.
> Use **File → Save a copy in Drive** (it goes to the *Colab Notebooks* folder of your ASE Google Drive) and work in that copy.
> For the team project, **File → Save a copy in GitHub** puts the notebook directly in your team's repository.
> Files you create while the notebook runs (CSV, charts) are also temporary: set `SAVE_TO_DRIVE = True` in the next cell to keep them.

In [ ]:
# Optional (Colab only): keep the files you create in Google Drive
SAVE_TO_DRIVE = False   # set to True, run this cell and allow access to your Drive

import os
OUT_DIR = '.'
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = '/content/drive/MyDrive/MFM/Chapter_15'
    os.makedirs(OUT_DIR, exist_ok=True)
print('Save your outputs to:', os.path.abspath(OUT_DIR))


In [ ]:
import os
import re
import sys
import json
import subprocess
try:
    import pysentiment2  # noqa: F401  (Loughran-McDonald word lists)
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pysentiment2'], check=True)
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


## Data and helpers

In [ ]:
REPO_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/data/market/'
FC_RAW = 'https://raw.githubusercontent.com/danpele/MFM/main/Quantlets/Ch_15/'
# local copies of the course data, if the notebook runs inside the repository
MARKET_DIR = next((os.path.join(d, 'data', 'market') for d in ('.', '..', '../..', '../../..')
                   if os.path.isdir(os.path.join(d, 'data', 'market'))), '')
FC_DIR = next((d for d in ('.', '..', 'Quantlets/Ch_15', '../Quantlets/Ch_15', '../../Quantlets/Ch_15')
               if os.path.exists(os.path.join(d, 'ch15_c3_headlines.csv.gz'))), None)
OUT_DIR = globals().get('OUT_DIR', '.')

# US stocks with FNSPID news and daily prices in the course data (Meta appears in FNSPID as FB until 2022)
TICKERS = {'AAPL': 'AAPL.US', 'MSFT': 'MSFT.US', 'AMZN': 'AMZN.US', 'NVDA': 'NVDA.US', 'TSLA': 'TSLA.US',
           'GOOGL': 'GOOGL.US', 'GOOG': 'GOOGL.US', 'META': 'META.US', 'FB': 'META.US', 'JPM': 'JPM.US',
           'BAC': 'BAC.US', 'C': 'C.US', 'GS': 'GS.US', 'MS': 'MS.US', 'WFC': 'WFC.US', 'CSCO': 'CSCO.US',
           'GME': 'GME.US', 'MSTR': 'MSTR.US', 'COIN': 'COIN.US'}


def load_csv(name, **kw):
    """A saved table of the chapter (FNSPID headlines, daily scores), local copy or the repository."""
    path = os.path.join(FC_DIR, name) if FC_DIR else FC_RAW + name
    return pd.read_csv(path, **kw)


def read_market(symbol):
    """Daily prices of one asset from the course data (local copy or the GitHub repository)."""
    fname = f'{symbol}.csv'
    path = os.path.join(MARKET_DIR, fname)
    src = path if os.path.exists(path) else REPO_RAW + fname
    return pd.read_csv(src, index_col='date', parse_dates=True).sort_index()


def stock_returns(symbols, start='2009-01-01'):
    """Simple daily close-to-close returns (%) on the adjusted price, for the stocks and SPY."""
    px = pd.concat({s: read_market(s)['adjusted_close'] for s in symbols}, axis=1).loc[start:]
    return 100 * px.pct_change(fill_method=None)


def assign_trading_day(ts, days):
    """FNSPID gives only the publication date: a headline dated d is assigned to the first trading day >= d."""
    d0 = ts.dt.tz_convert('UTC').dt.tz_localize(None).dt.normalize()
    days = pd.DatetimeIndex(days).sort_values()
    pos = days.searchsorted(d0.values)
    ok = pos < len(days)
    out = pd.Series(pd.NaT, index=ts.index)
    out[ok] = days[pos[ok]]
    return out


def load_dictionaries():
    """Loughran-McDonald word lists, as distributed with the pysentiment2 package."""
    import pysentiment2
    st = os.path.join(os.path.dirname(pysentiment2.__file__), 'static')
    lm = pd.read_csv(os.path.join(st, 'LM.csv'))
    lm['Word'] = lm['Word'].astype(str).str.upper()
    return {'LM_neg': set(lm.loc[lm['Negative'] > 0, 'Word']), 'LM_pos': set(lm.loc[lm['Positive'] > 0, 'Word'])}


## Style

In [ ]:
# Chart style: transparent background, legend below the plot
plt.rcParams['figure.facecolor'] = 'none'
plt.rcParams['axes.facecolor'] = 'none'
plt.rcParams['savefig.facecolor'] = 'none'
plt.rcParams['savefig.transparent'] = True
plt.rcParams['axes.grid'] = False
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.sans-serif'] = ['Helvetica', 'Arial', 'DejaVu Sans']
plt.rcParams['font.size'] = 9
plt.rcParams['axes.labelsize'] = 10
plt.rcParams['axes.titlesize'] = 10
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['legend.facecolor'] = 'none'
plt.rcParams['legend.framealpha'] = 0
plt.rcParams['legend.fontsize'] = 8

# Colours of Chapter 15
MainBlue, IDAred, Forest, Orange, Teal, Navy = '#1A3A6E', '#CD0000', '#2E7D32', '#E67E22', '#17A2B8', '#1F2A44'


def save_fig(name):
    """Save the figure as transparent PDF and PNG."""
    plt.savefig(os.path.join(OUT_DIR, f'{name}.pdf'), bbox_inches='tight', transparent=True)
    plt.savefig(os.path.join(OUT_DIR, f'{name}.png'), bbox_inches='tight', transparent=True, dpi=180)
    plt.show()
    plt.close()
    print(f'   saved {name}')


def fig_legend_bottom(fig, handles, labels, ncol=3, y=0.0):
    """One legend for the whole figure, below the panels."""
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)


## Robust MNIR word lists and the out-of-sample horse race

In [ ]:
PAPER = {'doi': '10.1016/j.jfineco.2022.11.006',
         # Table 2 (earnings calls, out of sample 2016-2020; controls and fixed effects), Section 3.1
         'r2_controls': 1.7, 'r2_lm': 2.1, 'r2_ml': 4.6, 'r2_overlap': 5.4, 'r2_bigrams': 4.5, 'r2_joint': 6.5,
         'b_lm': (0.41, -0.50), 'b_ml': (0.98, -1.37), 'b_overlap': (1.25, -1.56),
         # Section 3.2, full-sample dictionaries
         'n_overlap': (18, 30), 'n_ml_only': (57, 64), 'n_lm_only': (329, 2315)}

K, Q, P_VOCAB, CUT, SEED = 500, 5000, 2 ** 14, 80.0, 0
TRAIN_END = '2019-12-31'
CUTS = [50, 55, 60, 65, 70, 75, 80, 85, 90, 95]

# Snowball English stop-word list (snowballstem.org/algorithms/english/stop.txt)
SNOWBALL = """i me my myself we our ours ourselves you your yours yourself yourselves he him his himself she her hers
herself it its itself they them their theirs themselves what which who whom this that these those am is are was were
be been being have has had having do does did doing would should could ought i'm you're he's she's it's we're they're
i've you've we've they've i'd you'd he'd she'd we'd they'd i'll you'll he'll she'll we'll they'll isn't aren't wasn't
weren't hasn't haven't hadn't doesn't don't didn't won't wouldn't shan't shouldn't can't cannot couldn't mustn't let's
that's who's what's here's there's when's where's why's how's a an the and but if or because as until while of at by
for with about against between into through during before after above below to from up down in out on off over under
again further then once here there when where why how all any both each few more most other some such no nor not only
own same so than too very""".split()
# the paper (Appendix, note 47) keeps these stop words and adds can, will, must, let
KEEP = {'against', 'above', 'below', 'up', 'down', 'over', 'under', 'again', 'further', 'few', 'more', 'most',
        'no', 'not'}
STOP = (set(SNOWBALL) - KEEP) | set()
# proxy for the proper-noun filter: names and symbols of the 16 firms of the panel
FIRMS = set("""apple aapl microsoft msft amazon amzn nvidia nvda tesla tsla alphabet google googl goog meta facebook fb
jpmorgan jpm chase bac citigroup citi goldman sachs morgan stanley wells fargo wfc cisco csco gamestop gme
microstrategy mstr coinbase coin""".split())
CONTR = [(r"n't\b", ' not'), (r"'ll\b", ' will'), (r"'re\b", ' are'), (r"'d\b", ' would'), (r"'m\b", ' am'),
         (r"'ve\b", ' have'), (r"\bcannot\b", 'can not'), (r"'s\b", '')]
WORD = re.compile(r"[a-z]+")


def clean(text):
    t = str(text).lower().replace('’', "'")
    t = re.sub(r"[^\x00-\x7f]", ' ', t)
    for a, b in CONTR:
        t = re.sub(a, b, t)
    return [w for w in WORD.findall(t)
            if w not in STOP and w not in FIRMS and (len(w) > 2 or w in ('no', 'up', 'go'))]


def winsor(x, lo=0.01, hi=0.99):
    a, b = np.nanquantile(x, [lo, hi])
    return np.clip(x, a, b)


def cluster_ols_multi(y, X, groups):
    """OLS y = a + X b with day-clustered standard errors (as mfm_text.cluster_ols, several regressors)."""
    ok = X.std(0) > 0                      # a list without any word in the test texts has no coefficient
    if not ok.all():
        r = cluster_ols_multi(y, X[:, ok], groups) if ok.any() else {'b': [], 'se': [], 't': [], 'r2': 0.0,
                                                                      'r2_adj': 0.0, 'n': len(y), 'G': 0}
        full = {k: [float('nan')] * len(ok) for k in ('b', 'se', 't')}
        for k in full:
            vals = iter(r[k])
            full[k] = [next(vals) if o else None for o in ok]
        return {**r, **full}
    X = np.column_stack([np.ones(len(y)), X])
    XtX = np.linalg.inv(X.T @ X)
    b = XtX @ X.T @ y
    e = y - X @ b
    n, k = X.shape
    g = pd.factorize(np.asarray(groups))[0]
    S = np.zeros((g.max() + 1, k))
    np.add.at(S, g, X * e[:, None])
    G_ = S.shape[0]
    V = XtX @ (S.T @ S) @ XtX * G_ / (G_ - 1) * (n - 1) / (n - k)
    se = np.sqrt(np.diag(V))
    r2 = 1 - (e @ e) / np.sum((y - y.mean()) ** 2)
    r2a = 1 - (1 - r2) * (n - 1) / (n - k)
    return {'b': b[1:].tolist(), 'se': se[1:].tolist(), 't': (b[1:] / se[1:]).tolist(), 'r2': float(r2),
            'r2_adj': float(r2a), 'n': int(n), 'G': int(G_)}


def build_docs():
    h = load_csv('ch15_c3_headlines.csv.gz', dtype=str, keep_default_na=False)
    rets = stock_returns(sorted(set(TICKERS.values())) + ['SPY.US'], start='2009-01-01')
    rets.columns = [c.replace('.US', '') for c in rets.columns]
    h['day'] = assign_trading_day(pd.to_datetime(h['date'], utc=True), rets.index)
    h = h.dropna(subset=['day'])
    h = h[h['day'] >= '2009-06-01']
    docs = h.groupby(['day', 'ticker'])['headline'].apply(lambda s: ' '.join(s))
    n_hl = h.groupby(['day', 'ticker']).size()
    # 4-day buy-and-hold excess return: close d-1 to close d+2
    g = np.log1p(rets.loc[:'2023-12-31'] / 100)
    w = sum(g.shift(-k) for k in range(3))
    bh = np.expm1(w)
    ex = 100 * bh.drop(columns='SPY').sub(bh['SPY'], axis=0)
    R = ex.stack()
    R.index.names = ['day', 'ticker']
    D = pd.DataFrame({'text': docs, 'n': n_hl}).join(R.rename('R'), how='inner').dropna(subset=['R'])
    daily = load_csv('ch15_news_daily.csv', parse_dates=['day']).set_index(['day', 'ticker'])
    D = D.join(daily[['finbert', 'qwen', 'lm']], how='left')
    return D


def dtm(tok_lists, vocab):
    ix = {w: i for i, w in enumerate(vocab)}
    rows, cols = [], []
    for r, ws in enumerate(tok_lists):
        for w in ws:
            j = ix.get(w)
            if j is not None:
                rows.append(r)
                cols.append(j)
    C = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(tok_lists), len(vocab)))
    C.sum_duplicates()
    return C


def mnir_signs(C, v):
    """Sign of each word's loading in the Poisson inverse regression c_ij ~ Poisson(m_i exp(a_j + phi_j v_i)):
    score statistic at phi_j = 0 (a_j at its MLE), sign kept when z^2 > 2 (AIC), else 0."""
    m = np.asarray(C.sum(1)).ravel()
    keep = m > 0
    C, v, m = C[keep], v[keep], m[keep]
    v = (v - v.mean()) / v.std()
    Mtot = m.sum()
    p = np.asarray(C.sum(0)).ravel() / Mtot
    vbar = (m * v).sum() / Mtot
    score = C.T @ v - p * (m * v).sum()
    info = p * (m * (v - vbar) ** 2).sum()
    z = np.where(info > 0, score / np.sqrt(np.where(info > 0, info, 1)), 0.0)
    return np.where(z ** 2 > 2, np.sign(z), 0.0)


def robust_mnir(C, v, k=K, q=Q, seed=SEED):
    rng = np.random.default_rng(seed)
    n = C.shape[0]
    pos = np.zeros(C.shape[1])
    neg = np.zeros(C.shape[1])
    for _ in range(k):
        idx = rng.choice(n, size=q, replace=False)
        s = mnir_signs(C[idx], v[idx])
        pos += s > 0
        neg += s < 0
    return 100 * (pos - neg) / k


def scores(tok_lists, words):
    words = set(words)
    return np.array([sum(w in words for w in ws) / max(len(ws), 1) for ws in tok_lists])


def zscore(x):
    """Unit variance. Word shares are not winsorized: they lie in [0, 1] and are 0 for most firm-days, so the 1/99%
    winsorization of the paper would set the sparse lists to a constant."""
    s = x.std()
    return (x - x.mean()) / s if s > 0 else x * 0


def _r2(y, X):
    X = np.column_stack([np.ones(len(y)), X])
    b = np.linalg.lstsq(X, y, rcond=None)[0]
    e = y - X @ b
    return 1 - (e @ e) / np.sum((y - y.mean()) ** 2)


def boot_r2(y, groups, Xs, B=999, seed=SEED):
    """Day-cluster bootstrap (days drawn with replacement) of the R^2 differences between specifications."""
    rng = np.random.default_rng(seed)
    g = pd.factorize(np.asarray(groups))[0]
    rows = pd.Series(np.arange(len(y))).groupby(g).apply(np.array).tolist()
    pairs = [('ml_only', 'lm_all'), ('qwen', 'ml_only'), ('finbert', 'ml_only')]
    D = {f'{a}-{b}': [] for a, b in pairs}
    for _ in range(B):
        idx = np.concatenate([rows[i] for i in rng.integers(0, len(rows), len(rows))])
        r = {k: _r2(y[idx], X[idx]) for k, X in Xs.items() if X[idx].std(0).min() > 0}
        for a, b in pairs:
            if a in r and b in r:
                D[f'{a}-{b}'].append(r[a] - r[b])
    out = {}
    for k, v in D.items():
        v = 100 * np.array(v)
        a, b = k.split('-')
        out[k] = {'diff': 100 * float(_r2(y, Xs[a]) - _r2(y, Xs[b])), 'lo': float(np.quantile(v, 0.025)),
                  'hi': float(np.quantile(v, 0.975)), 'p_le0': float((v <= 0).mean()), 'B': len(v)}
    return out


def horse_race(Dt, toks, lists):
    """Out-of-sample regressions of the test period (Table 2 of the paper, without controls)."""
    y = winsor(Dt['R'].values)
    grp = Dt.index.get_level_values('day')
    S = {k: zscore(scores(toks, w)) for k, w in lists.items()}
    out = {}
    specs = {'lm_all': ['lm_all_pos', 'lm_all_neg'], 'lm_only': ['lm_only_pos', 'lm_only_neg'],
             'ml_only': ['ml_only_pos', 'ml_only_neg'], 'overlap': ['ov_pos', 'ov_neg'],
             'joint': ['lm_only_pos', 'lm_only_neg', 'ml_only_pos', 'ml_only_neg', 'ov_pos', 'ov_neg']}
    for name, cols in specs.items():
        out[name] = cluster_ols_multi(y, np.column_stack([S[c] for c in cols]), grp)
        out[name]['vars'] = cols
    for c in ('finbert', 'qwen'):
        out[c] = cluster_ols_multi(y, zscore(winsor(Dt[c].values))[:, None], grp)
        out[c]['vars'] = [c]
    out['boot'] = boot_r2(y, grp, {'lm_all': np.column_stack([S['lm_all_pos'], S['lm_all_neg']]),
                                   'ml_only': np.column_stack([S['ml_only_pos'], S['ml_only_neg']]),
                                   'qwen': zscore(winsor(Dt['qwen'].values))[:, None],
                                   'finbert': zscore(winsor(Dt['finbert'].values))[:, None]})
    out['share_hit'] = {k: float((scores(toks, w) > 0).mean()) for k, w in lists.items()}
    return out


def run():
    D = build_docs()
    toks = [clean(t) for t in D['text']]
    freq = pd.Series([w for ws in toks for w in ws]).value_counts()
    vocab = freq.index[:P_VOCAB].tolist()
    C = dtm(toks, vocab)
    day = D.index.get_level_values('day')
    tr = np.asarray(day <= TRAIN_END)
    te = ~tr
    v = winsor(D['R'].values)
    Dplus = robust_mnir(C[tr], v[tr])
    lm = load_dictionaries()
    LMp = {w.lower() for w in lm['LM_pos']}
    LMn = {w.lower() for w in lm['LM_neg']}
    toks_te = [t for t, b in zip(toks, te) if b]
    Dt = D[te]
    res = {'n_docs': int(len(D)), 'n_train': int(tr.sum()), 'n_test': int(te.sum()),
           'n_headlines': int(D['n'].sum()), 'n_vocab': len(vocab), 'n_types': int(len(freq)),
           'train': [str(day[tr].min().date()), str(day[tr].max().date())],
           'test': [str(day[te].min().date()), str(day[te].max().date())],
           'k': K, 'q': Q, 'cut': CUT, 'paper': PAPER, 'by_cut': {}}

    def lists_at(cut):
        MLp = {w for w, d in zip(vocab, Dplus) if d >= cut}
        MLn = {w for w, d in zip(vocab, Dplus) if d <= -cut}
        return MLp, MLn, {'lm_all_pos': LMp, 'lm_all_neg': LMn, 'lm_only_pos': LMp - MLp, 'lm_only_neg': LMn - MLn,
                          'ml_only_pos': MLp - LMp, 'ml_only_neg': MLn - LMn, 'ov_pos': LMp & MLp,
                          'ov_neg': LMn & MLn}

    for cut in CUTS:
        MLp, MLn, L = lists_at(cut)
        hr = horse_race(Dt, toks_te, L)
        res['by_cut'][str(cut)] = {'n_ml_pos': len(MLp), 'n_ml_neg': len(MLn),
                                   'n_ov_pos': len(L['ov_pos']), 'n_ov_neg': len(L['ov_neg']),
                                   'n_ml_only_pos': len(L['ml_only_pos']), 'n_ml_only_neg': len(L['ml_only_neg']),
                                   'reg': hr}
    MLp, MLn, L = lists_at(CUT)
    vs = pd.Series(Dplus, index=vocab)
    fr = freq.reindex(vocab)
    res['top_pos'] = [[w, float(vs[w]), int(fr[w])] for w in vs[vs >= CUT].sort_values(ascending=False).index[:40]]
    res['top_neg'] = [[w, float(vs[w]), int(fr[w])] for w in vs[vs <= -CUT].sort_values().index[:40]]
    res['dplus50'] = [[w, float(vs[w]), int(fr[w])] for w in vs[vs.abs() >= 50].sort_values(ascending=False).index]
    res['ml_pos'] = sorted(MLp)
    res['ml_neg'] = sorted(MLn)
    res['ov_pos'] = sorted(L['ov_pos'])
    res['ov_neg'] = sorted(L['ov_neg'])
    # LM words in the vocabulary that the algorithm colours the other way, and LM words it leaves neutral
    lm_in = [w for w in vocab if w in LMp | LMn]
    res['lm_in_vocab'] = len(lm_in)
    res['lm_flip'] = sorted([w for w in lm_in if (w in LMp and vs[w] <= -CUT) or (w in LMn and vs[w] >= CUT)])
    with open(os.path.join(OUT_DIR, 'ch15_colour_results.json'), 'w') as f:
        json.dump(res, f, indent=1)
    return res


def fig_colour_words(res):
    """Left: out-of-sample R^2 of each specification (2020-2023). Right: return per 1 s.d. of each word list."""
    from matplotlib.patches import Patch
    R = res['by_cut'][str(int(CUT))]['reg']
    names = [('lm_all', 'LM, all words'), ('lm_only', 'LM words not in ML'), ('ml_only', 'ML words not in LM'),
             ('overlap', 'Words in LM and ML'), ('joint', 'All three lists'), ('finbert', 'FinBERT'),
             ('qwen', 'Qwen2.5-7B')]
    col = {'lm_all': Orange, 'lm_only': Orange, 'ml_only': Teal, 'overlap': Teal, 'joint': Teal,
           'finbert': MainBlue, 'qwen': MainBlue}
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(7.2, 3.3), gridspec_kw={'width_ratios': [1.1, 1]})
    y = np.arange(len(names))[::-1]
    v = [100 * R[k]['r2'] for k, _ in names]
    a1.barh(y, v, 0.62, color=[col[k] for k, _ in names])
    for yi, vi in zip(y, v):
        a1.text(vi + 0.03, yi, f'{vi:.2f}', va='center', fontsize=7.5, color='black')
    a1.set_yticks(y)
    a1.set_yticklabels([lab for _, lab in names])
    a1.set_xlim(0, max(v) * 1.25)
    a1.set_xlabel('Out-of-sample $R^2$, 2020–2023 (%)')
    a1.set_title('How much of the 4-day return is explained?')
    grp = [('lm_only', 'LM not in ML'), ('ml_only', 'ML not in LM'), ('overlap', 'LM and ML')]
    x = np.arange(len(grp))
    for j, (sgn, c) in enumerate(((0, Forest), (1, IDAred))):
        b = [R[k]['b'][sgn] for k, _ in grp]
        e = [1.96 * R[k]['se'][sgn] for k, _ in grp]
        a2.bar(x + (j - 0.5) * 0.36, b, 0.36, color=c, yerr=e, capsize=3, ecolor=Navy)
    a2.axhline(0, color=Navy, lw=0.6)
    a2.set_xticks(x)
    a2.set_xticklabels([lab for _, lab in grp])
    a2.set_ylabel('Excess return per 1 s.d.\nof the word share (%)')
    a2.set_title('Positive and negative lists')
    fig.tight_layout()
    h = [Patch(color=Orange), Patch(color=Teal), Patch(color=MainBlue), Patch(color=Forest),
         Patch(color=IDAred)]
    fig_legend_bottom(fig, h, ['Loughran-McDonald (LM) lists', 'Lists learned from returns (ML)',
                                 'Language models (daily mean score)', 'Positive words (95% CI, day-clustered)',
                                 'Negative words (95% CI, day-clustered)'], ncol=3, y=0.0)
    save_fig('ch15_colour_words')


## Run

In [ ]:
r = run()
fig_colour_words(r)
c = r['by_cut'][str(int(CUT))]
print(f"Stock-days: {r['n_docs']} (training {r['n_train']}, {r['train'][0]} to {r['train'][1]}; test {r['n_test']}, "
      f"{r['test'][0]} to {r['test'][1]}); vocabulary {r['n_vocab']} of {r['n_types']} word types")
print(f"Cut-off |D+| >= {CUT:.0f}%: {c['n_ml_pos']} positive and {c['n_ml_neg']} negative ML words; "
      f"shared with LM: {r['ov_pos']} and {r['ov_neg']}")
rows = {k: dict(R2_pct=100 * v['r2'], **{f'b_{x}': b for x, b in zip(v['vars'], v['b'])})
        for k, v in c['reg'].items() if k not in ('boot', 'share_hit')}
print('Out-of-sample regressions, 2020-2023: R2 (%) and excess return (%) per 1 s.d. of each word share')
print(pd.DataFrame(rows).T.round(3).to_string())
print('Day-cluster bootstrap of R2 differences (percentage points):')
print(pd.DataFrame(c['reg']['boot']).T.round(3).to_string())
print('R2 (%) by cut-off:', {k: round(100 * v['reg']['ml_only']['r2'], 2) for k, v in r['by_cut'].items()})
print('Strongest positive words:', [w for w, *_ in r['top_pos'][:20]])
print('Strongest negative words:', [w for w, *_ in r['top_neg'][:20]])


![ch15_colour_words](./ch15_colour_words.png)

Output: `ch15_colour_words.pdf`, `ch15_colour_results.json`